# Corpus Scaling: Recall@k Before and After Full-Corpus Indexing

This notebook measures whether expanding from the 100-article test corpus to the full SEP corpus degrades retrieval quality, and by how much. The reasoning behind this measurement, including the external research that motivated it, is recorded in `scaling_and_pilot_strategy_log.md`.

**Why Recall@k and not similarity scores.** EnterpriseRAG-Bench ([arxiv.org/pdf/2605.05253](https://arxiv.org/pdf/2605.05253)) evaluated retrieval across five corpus sizes and found that as a corpus grows, top-k cosine similarity *rises* while Recall@k *declines*, for both BM25 and dense vector search. Watching similarity scores would therefore produce exactly the wrong conclusion. Recall@k asks a much more direct question: did the chunk that actually answers this question appear in the top k results at all? A related analysis of HNSW, the indexing algorithm Pinecone uses, makes the same point from the other direction: retrieval quality degrades silently as a vector database grows, with no errors and no latency change to signal it, which is why this has to be measured rather than eyeballed.

**Why naive single-query retrieval.** Multiquery decomposition fires several searches and pools the results, which partially masks crowding damage, arguably its whole benefit. That makes it a poor instrument for *measuring* crowding. Naive single-query gives the unmasked number.

**What counts as a hit, and an honest note on its limits.** This measures whether any chunk from the *expected article* appears in the top k. An earlier version of this notebook tried to match at section level, using the eval set's `Section_Expected` field, but that turned out not to work with the metadata currently indexed: the eval file's expected sections were written from article tables of contents and include subsections like "3.1 Thinking Animal Argument", while `section_parser.py` splits only at top-level headings, so the stored metadata carries only "Arguments for and Objections to Animalism". There is no subsection field to compare against, so section matching produced false misses on questions where retrieval had actually worked correctly.

Article-level matching is looser, and it's worth being clear about what that gives up: it cannot detect the Oruka-style failure, where the right article is retrieved but a wrong section within it outranks the correct one. That failure mode is real and documented in `oruka_retrieval_debug_log.md`, it just isn't measurable with the metadata as currently indexed. What article-level recall *does* measure directly is **cross-article crowding**, other articles pushing the correct one out of the results, which is precisely the risk the corpus-scaling research identifies. For the question this notebook exists to answer, it's the right instrument.

**Run order.** Part 1 establishes the baseline against the existing 100-article data, which lives in the index's default namespace. That runs *before* the full corpus is indexed, partly to have the number in hand independently, and partly as a checkpoint, since a baseline that already looks poor would be worth understanding before spending time indexing 1,800 articles. Part 2 runs the identical measurement against the full corpus once it's been indexed into its own namespace, and Part 3 compares them directly.


In [1]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())


Working directory set to: c:\Users\user\Dropbox\Career\Tech\++Projects\DeepAnalytic\SEP


In [2]:
import pandas as pd

from config import settings
from embeddings import get_embedder
from vectorstore import VectorDB

embed = get_embedder()
vector_db = VectorDB()
index = vector_db.connect_to_index(settings.PINECONE_INDEX_NAME)

# Namespaces. The existing 100-article data sits in the default namespace
# (passed as None). The full corpus goes into its own named namespace.
NAMESPACE_100 = None
NAMESPACE_FULL = "articles-full"

SEARCH_K = 50   # retrieve wide, so rank position is visible for chunks that miss top-10
REPORT_KS = [5, 10, 20]

print(f"Index: {settings.PINECONE_INDEX_NAME}")


Index: sep-articles-intros-added


c:\Users\user\anaconda3\envs\deepanalytic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the eval questions

Pulled from `tests/eval_systematic.csv`, deduplicated to one row per question (the file contains each question twice, once for naive mode and once for rerank).


In [3]:
eval_df = pd.read_csv("tests/eval_systematic.csv")

questions = (
    eval_df[["Article", "Question", "Section_Expected"]]
    .drop_duplicates(subset=["Question"])
    .reset_index(drop=True)
)

print(f"{len(questions)} unique questions loaded\n")
for i, row in questions.iterrows():
    print(f"{i+1}. [{row['Article']}] {row['Question']}")
    print(f"     expected section: {row['Section_Expected']}")


10 unique questions loaded

1. [Peter Abelard] Why does Abelard reject the theory that universals are real things?
     expected section: 2. Metaphysics
2. [Peter Abelard] How does Abelard's theory of intentions determine moral worth?
     expected section: 6. Ethics
3. [African Sage Philosophy] What three negative claims about African philosophy was Oruka trying to counter?
     expected section: 1. Oruka's Project
4. [African Sage Philosophy] What distinguishes a folk sage from a philosophic sage?
     expected section: 5. What counts as Sage Philosophy
5. [al-Farabi] What is al-Farabi's Lesser Harmony project?
     expected section: none - trap question
6. [al-Farabi] Why does al-Farabi consider metaphysics not a theological science?
     expected section: 6. Metaphysics
7. [Animalism] What is the thinking animal argument?
     expected section: 3.1 Thinking Animal Argument
8. [Animalism] What's the difference between organic and somatic animalism?
     expected section: 1.2 Our Per

## The measurement function

For each question: embed it, query the given namespace at k=50, then walk the ranked results looking for the first chunk from the expected article. Records that rank, or `None` if no chunk from that article appears anywhere in the top 50.

Alongside the rank, it records how many of the top 10 results came from an article *other* than the expected one. That second number is the more informative one for this particular investigation, since it measures crowding directly rather than as a pass/fail threshold.


In [4]:
def find_correct_rank(question: str, expected_article: str, expected_section: str, namespace):
    """
    Returns (rank_of_first_matching_chunk, list_of_top_results).
    Rank is 1-indexed. Returns None if no chunk from the expected article
    appears in the top SEARCH_K.

    Matches on ARTICLE only, not section. The eval CSV's Section_Expected
    values come from article TOCs and include subsections (e.g. "3.1 Thinking
    Animal Argument"), but section_parser.py only splits at top-level headings,
    so indexed metadata carries only top-level titles ("Arguments for and
    Objections to Animalism"). Section-level matching therefore produced false
    misses on questions where retrieval had actually worked correctly --
    expected_section is kept in the signature for reference but deliberately
    unused. See the note above on what this trade-off gives up.
    """
    qv = embed.embed_query(question)

    kwargs = {"vector": qv, "top_k": SEARCH_K, "include_metadata": True}
    if namespace:
        kwargs["namespace"] = namespace

    res = index.query(**kwargs)
    matches = res["matches"]

    found_rank = None
    for rank, m in enumerate(matches, 1):
        if m["metadata"].get("title") == expected_article:
            found_rank = rank
            break

    return found_rank, matches


def measure_namespace(namespace, label):
    """Run every eval question against one namespace and return a results dataframe."""
    rows = []
    for _, q in questions.iterrows():
        rank, matches = find_correct_rank(
            q["Question"], q["Article"], q["Section_Expected"], namespace
        )

        # How many of the top 10 came from an article other than the expected one --
        # a direct measure of cross-article crowding.
        top10_titles = [m["metadata"].get("title") for m in matches[:10]]
        off_article = sum(1 for t in top10_titles if t != q["Article"])

        rows.append({
            "Article": q["Article"],
            "Question": q["Question"][:60] + "...",
            "Rank": rank,
            "Hit@5": rank is not None and rank <= 5,
            "Hit@10": rank is not None and rank <= 10,
            "Hit@20": rank is not None and rank <= 20,
            "OffArticle@10": off_article,
        })

    df = pd.DataFrame(rows)
    print(f"=== {label} ===")
    print(df[["Article", "Rank", "Hit@10", "OffArticle@10"]].to_string(index=False))
    print()
    for k in REPORT_KS:
        recall = df[f"Hit@{k}"].mean()
        print(f"  Recall@{k}: {recall:.2f}")
    found = df["Rank"].notna()
    if found.any():
        print(f"  Mean rank (where found): {df.loc[found, 'Rank'].mean():.1f}")
    print(f"  Never found in top {SEARCH_K}: {(~found).sum()} of {len(df)}")
    print(f"  Mean off-article chunks in top 10: {df['OffArticle@10'].mean():.1f}")
    return df


## Part 1: Baseline against the 100-article corpus

Run this before indexing the full corpus. If several questions already miss at k=10 here, that's worth understanding before scaling up, since it would mean the pipeline has a retrieval problem independent of corpus size.


In [5]:
results_100 = measure_namespace(NAMESPACE_100, "100-article corpus (default namespace)")


=== 100-article corpus (default namespace) ===
                     Article  Rank  Hit@10  OffArticle@10
               Peter Abelard     1    True              3
               Peter Abelard     1    True              3
     African Sage Philosophy     1    True              2
     African Sage Philosophy     1    True              0
                   al-Farabi     6    True              8
                   al-Farabi     3    True              8
                   Animalism     1    True              0
                   Animalism     1    True              0
Ancient Political Philosophy     1    True              8
Ancient Political Philosophy     1    True              0

  Recall@5: 0.90
  Recall@10: 1.00
  Recall@20: 1.00
  Mean rank (where found): 1.7
  Never found in top 50: 0 of 10
  Mean off-article chunks in top 10: 3.2


### Baseline: the pipeline retrieves cleanly at 100 articles

Recall@10 and @20 are both 1.00, every question surfaced its expected article, and nothing was missing from the top fifty. Mean rank is 1.7, with eight of ten questions putting the correct article at position one. This lines up with the earlier human-scored evaluation that rated naive retrieval 4.11 out of 5 on these same questions, which is a useful independent check that the measurement is behaving sensibly.

Two things shape how the full-corpus comparison should be read. Recall@10 is already at ceiling, so it can only stay flat or fall, making it a good detector of serious breakage and a poor one for anything subtler. And crowding is already present: the mean off-article count is 3.2, so roughly a third of every top-ten result set comes from an article other than the one asked about. That number has room to move in both directions, and it is the more informative of the two.

The average hides an uneven distribution. Four questions have zero off-article chunks, while both al-Farabi questions and one Ancient Political Philosophy question sit at 8 out of 10. Those three have the least headroom, and if expansion pushes anything out of the top ten, it will likely be one of them first.

## Part 2: Full corpus

Run this only after `ingest.py` has completed with `TEST_MODE = False` and `NAMESPACE = "articles-full"`.


In [6]:
results_full = measure_namespace(NAMESPACE_FULL, "Full SEP corpus (articles-full namespace)")


=== Full SEP corpus (articles-full namespace) ===
                     Article  Rank  Hit@10  OffArticle@10
               Peter Abelard     3    True              7
               Peter Abelard     1    True              5
     African Sage Philosophy     1    True              2
     African Sage Philosophy     1    True              4
                   al-Farabi     6    True              9
                   al-Farabi     4    True              8
                   Animalism     1    True              1
                   Animalism     1    True              0
Ancient Political Philosophy     3    True              8
Ancient Political Philosophy     1    True              3

  Recall@5: 0.90
  Recall@10: 1.00
  Recall@20: 1.00
  Mean rank (where found): 2.2
  Never found in top 50: 0 of 10
  Mean off-article chunks in top 10: 4.7


### Full corpus: recall holds, result sets diversify

Every recall figure is identical to the baseline. Recall@5 stays at 0.90, Recall@10 and @20 stay at 1.00, nothing dropped out of the top fifty. A 19.5x expansion, from 5,687 chunks to 111,048, cost nothing in recall on this set of questions.

Two other numbers moved. Mean rank slipped from 1.7 to 2.2, and mean off-article chunks rose from 3.2 to 4.7. Six of ten questions saw their off-article count rise, with the two Peter Abelard questions moving most sharply, from 3 to 7 and from 3 to 5.

**What that second number does not tell us is whether the change is bad.** The metric counts chunks from *other* articles, not necessarily from *wrong* ones. The 100-article slice was alphabetical, so it excluded most of the corpus by construction, including entries that are genuinely relevant to these questions. A question about al-Farabi's metaphysics now competing with an article on Arabic and Islamic metaphysics is not necessarily noise displacing signal, it is the corpus finally containing material that should have been there. Crowding by irrelevant content and enrichment by relevant content produce identical numbers here.

Distinguishing them requires reading what actually came back, which is what the head-query section below does. For now the honest summary is narrower than it first appears: recall held, result sets became more diverse, and whether that diversity is noise or substance is not yet established.

## Part 3: Direct comparison

The number that matters is the change in Recall@10 and in rank position. A question that stays in the top 10 but slides from rank 2 to rank 9 hasn't failed yet, but it's heading that way, so rank movement is worth watching alongside the pass/fail counts.

The `OffArticle` columns carry the mechanism this whole comparison exists to detect. Xiang et al. ([arxiv.org/pdf/2506.05690](https://arxiv.org/pdf/2506.05690)) measured vector RAG accuracy on complex reasoning dropping from 58.6% to 43.2% as a corpus grew 20x, and named the cause directly: vector retrieval "is prone to capturing high-similarity but irrelevant noise as the search space expands." This expansion is roughly 19.5x, close to the same scale factor, so if that effect is real here it should show up as more off-article chunks crowding the top 10. Worth holding a counterweight alongside it, though: "Less LLM, More Documents" ([arxiv.org/html/2510.02657](https://arxiv.org/html/2510.02657)) found corpus scaling consistently *strengthens* RAG on open-domain factual QA, with the degradation concentrated on complex reasoning tasks. Both findings are real and apply to different task types.

The RAG evaluation checklist at [hiro.solutions](https://hiro.solutions/rag-evaluation-checklist-retrieval-quality-answer-accuracy) also advises tracking exactly this kind of turnover rather than aggregate scores: "review lost documents. Which previously retrievable sources disappeared from top-k? This is often more informative than average score changes."


In [7]:
comparison = pd.DataFrame({
    "Article": results_100["Article"],
    "Rank_100": results_100["Rank"],
    "Rank_full": results_full["Rank"],
    "Hit@10_100": results_100["Hit@10"],
    "Hit@10_full": results_full["Hit@10"],
    "OffArticle_100": results_100["OffArticle@10"],
    "OffArticle_full": results_full["OffArticle@10"],
})

comparison["Rank_change"] = comparison["Rank_full"] - comparison["Rank_100"]
comparison["Newly_missing"] = comparison["Hit@10_100"] & ~comparison["Hit@10_full"]

print(comparison.to_string(index=False))
print()
for k in REPORT_KS:
    r100 = results_100[f"Hit@{k}"].mean()
    rfull = results_full[f"Hit@{k}"].mean()
    print(f"Recall@{k}:  {r100:.2f} -> {rfull:.2f}  ({rfull - r100:+.2f})")

print()
print(f"Questions that dropped out of top 10: {comparison['Newly_missing'].sum()}")
print(f"Mean off-article chunks in top 10: {results_100['OffArticle@10'].mean():.1f} -> {results_full['OffArticle@10'].mean():.1f}")


                     Article  Rank_100  Rank_full  Hit@10_100  Hit@10_full  OffArticle_100  OffArticle_full  Rank_change  Newly_missing
               Peter Abelard         1          3        True         True               3                7            2          False
               Peter Abelard         1          1        True         True               3                5            0          False
     African Sage Philosophy         1          1        True         True               2                2            0          False
     African Sage Philosophy         1          1        True         True               0                4            0          False
                   al-Farabi         6          6        True         True               8                9            0          False
                   al-Farabi         3          4        True         True               8                8            1          False
                   Animalism         1          

### Side by side: nothing broke, and rank movement was minimal

The comparison confirms what the two runs suggested separately. No question dropped out of the top ten, no recall figure moved at any of the three thresholds, and rank changes were small: seven questions held their position exactly, two moved by 2, one moved by 1. Nothing regressed in a way that would affect what a user actually receives.

The al-Farabi pair is worth a specific note, since they were flagged at baseline as the least robust cases. They held. Ranks 6 and 3 became 6 and 4, and their off-article counts, already the highest in the set at 8 and 8, moved barely at all, to 9 and 8. The questions with the least headroom absorbed the expansion without slipping. That is more reassuring than the aggregate numbers alone, because these were the ones most likely to fail first.

The off-article column moved on six of ten questions, but per the previous note, that column cannot distinguish irrelevant crowding from genuine enrichment. Two cases make the ambiguity concrete. The second African Sage Philosophy question went from 0 to 4 off-article chunks while holding rank 1, and the second Ancient Political Philosophy question went from 0 to 3, also holding rank 1. Both kept the correct article at the top while pulling in additional material that the alphabetical 100-article slice had excluded. Whether that material is useful context or dilution is not something these numbers can answer.

What can be said cleanly: the expansion did not cost recall, did not push anything out of reach, and did not break the weakest cases. What remains open is whether the newly retrieved material is worth having, which the head-query section below is designed to probe.

## Part 4: Head queries -- generic concepts with no single correct article

The test above has a specific and now-identified weakness: **all ten questions are tail queries**. They target distinctive entities like Oruka, al-Farabi, and Abelard, which have few competitors even at 1,803 articles. The RAG evaluation literature is direct about this risk. The checklist at [hiro.solutions](https://hiro.solutions/rag-evaluation-checklist-retrieval-quality-answer-accuracy) advises that head and tail queries be checked separately because "improvements on common queries can hide regressions on niche but important questions." The inverse applies here just as much: a clean result on ten tail queries can hide a regression on the common ones.

Head queries are generic concepts that many articles discuss in passing. At 100 articles, a question about supervenience or the is-ought gap has few competitors. At 1,803, it competes against dozens of articles that each mention it. That is precisely the crowding mechanism this expansion was measured against, and the current test set cannot see it at all.

**These questions have no ground truth, and that's fine.** There is no single correct article for "what is supervenience," so Recall@k is not measurable. What is measurable, and what actually answers the question, is how much the retrieved result set *changes* between the two corpora. A question whose top results stay broadly stable has not been disrupted by crowding. One whose results turn over almost entirely has been, and the full-corpus results then need reading directly to judge whether they are still coherent or merely scattered.


In [9]:
HEAD_QUERIES = [
    "What is supervenience?",
    "What is the difference between necessary and sufficient conditions?",
    "What is the is-ought gap?",
    "What is the distinction between a priori and a posteriori knowledge?",
    "What is meant by natural kinds?",
]


def compare_head_query(question, k=10):
    """
    Run one generic question against both namespaces and compare the
    result sets. No ground truth, so this measures overlap and turnover
    rather than recall.
    """
    qv = embed.embed_query(question)

    res_100 = index.query(vector=qv, top_k=k, include_metadata=True)
    res_full = index.query(vector=qv, top_k=k, include_metadata=True, namespace=NAMESPACE_FULL)

    titles_100 = [m["metadata"].get("title") for m in res_100["matches"]]
    titles_full = [m["metadata"].get("title") for m in res_full["matches"]]

    # How many of the 100-corpus articles survived into the full-corpus top-k
    overlap = len(set(titles_100) & set(titles_full))

    # Distinct articles represented -- a proxy for whether results are
    # focused on a few relevant entries or scattered across many
    distinct_100 = len(set(titles_100))
    distinct_full = len(set(titles_full))

    return {
        "Question": question[:45] + "...",
        "Overlap": overlap,
        "Distinct_100": distinct_100,
        "Distinct_full": distinct_full,
        "Titles_100": titles_100,
        "Titles_full": titles_full,
    }


head_results = [compare_head_query(q) for q in HEAD_QUERIES]
head_df = pd.DataFrame(head_results)[["Question", "Overlap", "Distinct_100", "Distinct_full"]]

print(head_df.to_string(index=False))
print()
print(f"Mean articles surviving from 100-corpus top-10 into full-corpus top-10: {head_df['Overlap'].mean():.1f} of 10")
print(f"Mean distinct articles in top-10:  {head_df['Distinct_100'].mean():.1f} (100-corpus) -> {head_df['Distinct_full'].mean():.1f} (full)")


                                        Question  Overlap  Distinct_100  Distinct_full
                       What is supervenience?...        0             2              4
What is the difference between necessary and ...        0             5              2
                    What is the is-ought gap?...        0             6              8
What is the distinction between a priori and ...        1             2              7
              What is meant by natural kinds?...        1             7              3

Mean articles surviving from 100-corpus top-10 into full-corpus top-10: 0.4 of 10
Mean distinct articles in top-10:  4.4 (100-corpus) -> 4.8 (full)


### Head queries: near-total turnover, which is the expected result

The overlap numbers are striking. On average only 0.4 of the ten articles retrieved from the 100-article corpus survived into the full-corpus top ten. Three of the five questions had zero overlap at all, meaning the two corpora returned entirely disjoint result sets for the same question.

**This is not evidence of degradation, and reading it that way would be a mistake.** These are generic concepts, and the 100-article slice was alphabetical. There is no reason to expect the best articles on supervenience or the is-ought gap to fall in the first hundred entries alphabetically. At 100 articles the system was returning whatever happened to mention the concept in passing; at 1,803 it can return articles actually about it. Near-total turnover is what improvement looks like here, not what failure looks like.

The distinct-article counts also move in no consistent direction, which is informative in itself. Two questions returned *more* distinct articles on the full corpus (2 to 4, and 2 to 7), two returned *fewer* (5 to 2, and 7 to 3), and one rose slightly. If crowding were dominating, results would consistently scatter across more articles. Instead the pattern looks like each question finding whatever concentration of relevant material the corpus actually contains, which for a well-covered concept means a few substantial articles rather than many passing mentions.

None of this can be settled by the counts alone, though. Zero overlap is compatible with both "found much better sources" and "wandered off into loosely related material," and the numbers look identical either way. The side-by-side listing below is the only way to tell which happened.

### Reading the head-query results directly

Overlap counts alone don't settle whether the full corpus is doing better or worse, since a generic question *should* pull in more articles once more relevant articles exist. The judgement call requires actually reading what came back. The cell below prints both result sets side by side for each question.

Two things worth looking for. First, whether the full-corpus results are still recognisably about the concept asked, or whether they have drifted into articles that merely mention it in passing. Second, whether the results are dominated by a handful of articles each contributing several chunks, which the same [hiro.solutions checklist](https://hiro.solutions/rag-evaluation-checklist-retrieval-quality-answer-accuracy) flags as its own distinct problem, since "a retriever that returns five similar chunks may score acceptably on relevance but still starve the generator of useful context diversity."


In [10]:
for r in head_results:
    print("=" * 90)
    print(r["Question"])
    print("=" * 90)
    print("  100-corpus top 10:")
    for t in r["Titles_100"]:
        print(f"    - {t}")
    print("  Full-corpus top 10:")
    for t in r["Titles_full"]:
        print(f"    - {t}")
    print()


What is supervenience?...
  100-corpus top 10:
    - Anomalous Monism
    - Anomalous Monism
    - Anomalous Monism
    - Anomalous Monism
    - Anomalous Monism
    - Anomalous Monism
    - Anomalous Monism
    - Anomalous Monism
    - Aesthetic Judgment
    - Anomalous Monism
  Full-corpus top 10:
    - Ontological Dependence
    - Supervenience
    - Supervenience
    - Supervenience
    - Supervenience
    - Supervenience
    - Scientific Reduction
    - Supervenience
    - Moral Naturalism
    - Supervenience

What is the difference between necessary and ...
  100-corpus top 10:
    - Abstract Objects
    - Abstract Objects
    - Abilities
    - Causation in Arabic and Islamic Thought
    - Isaac Albalag
    - Arabic and Islamic Philosophy of Language and Logic
    - Abstract Objects
    - Abilities
    - Abilities
    - Abilities
  Full-corpus top 10:
    - Necessary and Sufficient Conditions
    - Fatalism
    - Necessary and Sufficient Conditions
    - Necessary and Sufficient 

### Reading the results: the turnover was improvement, not degradation

The titles settle what the counts could not. In four of five cases the full corpus returned the entry actually named after the concept, while the 100-article slice returned articles that merely mentioned it in passing.

"What is supervenience?" is the clearest case. The 100-article corpus returned *Anomalous Monism* nine times out of ten, which discusses supervenience but is not about it. The full corpus returned the *Supervenience* entry six times, alongside *Ontological Dependence*, *Scientific Reduction*, and *Moral Naturalism*, all genuinely adjacent. "What is meant by natural kinds?" shows the same pattern even more starkly: a scattered set including *Peter Abelard*, *Anaxagoras*, and *The Concept of the Aesthetic* became seven chunks from *Natural Kinds* plus *Natural Properties*. And "necessary and sufficient conditions" went from *Abstract Objects* and *Abilities* to nine chunks from *Necessary and Sufficient Conditions*.

The a priori question is the interesting middle case. The 100-article corpus already had the right article and returned it eight times. The full corpus returned it only three times, adding *Epistemology*, *The Epistemology of Modality*, *Kant's Theory of Judgment*, and *A Priorism in Moral Epistemology*. This is the one case where the earlier caution genuinely applies: results are more diverse and less concentrated, and whether that helps or dilutes depends on what the user wanted. For a broad orienting question the additional context is probably useful; for a precise definitional one the 100-article behaviour was arguably tighter.

The is-ought question is the weakest result and worth flagging. Neither corpus found a dedicated entry, which likely reflects the concept being distributed across many articles in SEP rather than having its own. The full corpus returned *Thick Ethical Concepts*, *Richard Mervyn Hare*, and *Supererogation*, which are reasonable, alongside *Combining Logics* and *Bernard Bolzano*, which look like genuine noise. This is the one case that resembles crowding rather than enrichment.

### Reading actual chunk text for the two ambiguous cases

Titles settled three of the five head queries cleanly: supervenience, natural kinds, and necessary/sufficient conditions all moved from articles that merely mention the concept to the dedicated entries named after it. No further checking is needed there.

Two cases genuinely warrant reading the passages. The is-ought result is the only one that resembles crowding rather than enrichment, and the judgement that *Combining Logics* and *Bernard Bolzano* look like noise was inferred from titles alone, not verified. The a priori result is the more decision-relevant of the two: the 100-article corpus returned the right article eight times out of ten, while the full corpus returned it three times alongside four adjacent entries. Whether that constitutes useful breadth or dilution cannot be read off titles, and it is also the case that most resembles a realistic user question, since both corpora had the correct article available.

In [11]:
AMBIGUOUS_QUERIES = [
    "What is the is-ought gap?",
    "What is the distinction between a priori and a posteriori knowledge?",
]

SNIPPET_CHARS = 350


def show_chunks(question, namespace, label, k=10):
    qv = embed.embed_query(question)
    kwargs = {"vector": qv, "top_k": k, "include_metadata": True}
    if namespace:
        kwargs["namespace"] = namespace
    res = index.query(**kwargs)

    print(f"--- {label} ---")
    for rank, m in enumerate(res["matches"], 1):
        meta = m["metadata"]
        text = (meta.get("text") or "").replace("\n", " ")[:SNIPPET_CHARS]
        print(f"[{rank}] {meta.get('title')} | {meta.get('section')}  (score {m['score']:.3f})")
        print(f"    {text}...")
    print()


for q in AMBIGUOUS_QUERIES:
    print("=" * 95)
    print(q)
    print("=" * 95)
    show_chunks(q, NAMESPACE_100, "100-article corpus")
    show_chunks(q, NAMESPACE_FULL, "full corpus")

What is the is-ought gap?
--- 100-article corpus ---
[1] Actualism and Possibilism in Ethics | Historical Origins of the Debate  (score 0.511)
    The historical origins of the debate may be traced back to work by Lars Bergström and Hector-Neri Castañeda. In his A Problem for Utilitarianism (1968), Castañeda argues that, given a few standard assumptions, utilitarianism is formally incoherent. His argument may be stated rather straightforwardly. First, Castañeda assumes a principle of deontic ...
[2] Actualism and Possibilism in Ethics | Actualism  (score 0.472)
    The relativization of obligations to different sets of options has led Jackson and Pargetter to reject the “ought distributes over conjunction” (ODC) principle (1986: 247). Recall that ODC holds that if an agent S ought to do both A and B, then S ought to do A and S ought to do B (Castañeda 1968: 141). While they accept that Procrastinate ought to ...
[3] Aquinas’ Moral, Political, and Legal Philosophy | Virtues  (score 0.44

### What to look for

For the is-ought question, the thing to check is whether the *Combining Logics* and *Bernard Bolzano* chunks actually discuss deriving normative claims from descriptive ones, or whether they surfaced on some incidental lexical or semantic overlap. If they contain real is-ought content, the earlier read was too harsh and this case belongs with the improvements rather than apart from them.

For the a priori question, the comparison is between concentration and breadth. Eight chunks from one article gives depth on a single treatment; three chunks from that article plus passages from *Epistemology*, *The Epistemology of Modality*, and *Kant's Theory of Judgment* gives a wider view at the cost of depth. Worth asking which set would actually produce a better answer if handed to the generation step, since that, rather than any count, is what the retrieval exists to serve.

### Reading the chunks: both cases resolve in favour of the full corpus

**The is-ought result was better than the titles suggested, and my earlier read of it was too harsh.** The 100-article corpus returned nothing about the is-ought gap at all. Its top hits are about actualism versus possibilism, deontic logic principles, and Kant's judgment of taste, matching on the word "ought" in unrelated technical contexts rather than on the concept. The full corpus led with *Thick Ethical Concepts*, which opens on exactly the right passage: "the intuitive contrast between is and ought marks an important gap between distinct domains, and sometimes this gap is identified as a distinction between facts and values." A second chunk from the same article addresses the same topic directly.

The two entries I had flagged as probable noise turn out to be defensible. *Combining Logics* is discussing the logical formalisation of ought-propositions and mixed deontic inference, and *Bernard Bolzano* covers his early anticipation of deontic logic. Neither is about the is-ought gap in the Humean sense, but both are genuinely about the logic of ought rather than incidental keyword matches. The scores support this reading too: the full corpus tops out at 0.632 against 0.511 for the 100-article corpus, and every full-corpus result scores above the best 100-article result.

**The a priori case resolves in favour of breadth rather than dilution.** The concern was that dropping from eight chunks of the correct article to three might cost depth. Reading the additions, it doesn't. *Epistemology* takes the top slot with a passage directly defining what counts as experience for a priori justification, scoring 0.709 against the 100-corpus best of roughly 0.68. *Kant's Theory of Judgment* covers the synthetic a priori, which is central to the distinction and absent from the 100-corpus results entirely. *A Priorism in Moral Epistemology* opens with a clean statement of the standard view.

What the 100-article corpus was doing instead is visible in its own results: six of its ten chunks come from a single section titled "What is a priori knowledge?", and several are edge cases rather than the distinction itself, one on the Lottery Paradox, one on Gareth Evans on contingent a priori propositions, one on John Turri's unlikely-event example. That is depth on peripheral debates, not depth on the question asked. The full corpus trades some of that for the definitional core plus Kant, which is a better set for the question.

<div style="color:green">

**Both ambiguous cases resolve the same way the three clear ones did.** All five head queries improved, and the near-total result-set turnover flagged earlier was the corpus finally surfacing the right material rather than crowding displacing it.
</div>

## Part 5: Paraphrase stability

The two al-Farabi questions in the main test returned the correct article at ranks 6 and 4, with 8 and 9 of their top 10 results coming from other articles. They passed, but narrowly, and that raises a fair question about whether they passed robustly or merely got lucky with phrasing.

This tests the same underlying question asked three different ways and checks whether the correct article still surfaces each time. Query-diversity guidance makes this point generally. Latenode's evaluation guide ([latenode.com](https://latenode.com/blog/rag-evaluation-complete-guide-to-testing-retrieval-augmented-generation-systems)) argues that while some systems handle structured, straightforward queries well, "they may falter when faced with conversational language, typos, or industry-specific terminology," so testing variations in language and complexity belongs in a serious evaluation rather than being an optional extra.


In [12]:
PARAPHRASE_SETS = [
    {
        "expected_article": "al-Farabi",
        "variants": [
            "Why does al-Farabi consider metaphysics not a theological science?",
            "How did al-Farabi distinguish metaphysics from theology?",
            "What did al-Farabi think metaphysics was actually about, if not God?",
        ],
    },
    {
        "expected_article": "Ancient Political Philosophy",
        "variants": [
            "Why does Cicero say Rome under the Republic satisfies the definition of a res publica?",
            "What made Rome a genuine commonwealth in Cicero's view?",
            "On what grounds did Cicero call the Roman Republic a res publica?",
        ],
    },
]


def check_paraphrases(expected_article, variants, namespace, label):
    print(f"--- {label} | expected: {expected_article} ---")
    for v in variants:
        rank, matches = find_correct_rank(v, expected_article, None, namespace)
        off_article = sum(1 for m in matches[:10] if m["metadata"].get("title") != expected_article)
        rank_str = str(rank) if rank else f"not in top {SEARCH_K}"
        print(f"  rank {rank_str:>18}  |  off-article in top 10: {off_article}  |  {v[:60]}")
    print()


for pset in PARAPHRASE_SETS:
    check_paraphrases(pset["expected_article"], pset["variants"], NAMESPACE_100, "100-corpus")
    check_paraphrases(pset["expected_article"], pset["variants"], NAMESPACE_FULL, "full corpus")


--- 100-corpus | expected: al-Farabi ---
  rank                  3  |  off-article in top 10: 8  |  Why does al-Farabi consider metaphysics not a theological sc
  rank                  4  |  off-article in top 10: 8  |  How did al-Farabi distinguish metaphysics from theology?
  rank                  6  |  off-article in top 10: 9  |  What did al-Farabi think metaphysics was actually about, if 

--- full corpus | expected: al-Farabi ---
  rank                  4  |  off-article in top 10: 8  |  Why does al-Farabi consider metaphysics not a theological sc
  rank                  4  |  off-article in top 10: 8  |  How did al-Farabi distinguish metaphysics from theology?
  rank                  6  |  off-article in top 10: 9  |  What did al-Farabi think metaphysics was actually about, if 

--- 100-corpus | expected: Ancient Political Philosophy ---
  rank                  1  |  off-article in top 10: 0  |  Why does Cicero say Rome under the Republic satisfies the de
  rank                 

### What would count as a problem here

Rank varying somewhat across phrasings is normal and not by itself concerning. What would be concerning is a phrasing that drops out of the top 10 entirely on the full corpus while holding on the 100-article corpus, since that would mean the expansion made retrieval genuinely fragile to wording rather than merely slightly noisier. Worth watching the al-Farabi set in particular, given it was already the weakest case in the main test.


### Paraphrase stability: rewording changes little, and expansion changes less

Both sets held across all three phrasings and both corpora. Nothing dropped out of the top ten anywhere, which was the specific failure this test was built to catch.

The al-Farabi set was the one under suspicion, since it was the weakest case in the main comparison. It behaved consistently rather than fragilely. Ranks landed at 3, 4, and 6 on the 100-article corpus and 4, 4, and 6 on the full corpus, so the only movement attributable to expansion was a single position on one phrasing. Off-article counts were identical across corpora at 8, 8, and 9. Notably, rewording moved rank more than expanding the corpus did: the spread across three phrasings on a single corpus is wider than the difference between corpora for any single phrasing. Whatever makes this question harder than the others is a property of the question itself, not of corpus size.

The Cicero set held rank 1 across all six runs. The only change is off-article counts rising from 0 to 3, 4, and 3, which is the same enrichment-versus-crowding ambiguity noted earlier. Given the head-query results resolved that ambiguity in favour of enrichment in five of five cases, and given rank 1 held throughout, the more likely reading is that the full corpus is surfacing genuinely relevant neighbouring material rather than displacing anything.

**Taken together with the head queries, the case for shipping the full corpus is now reasonably firm.** Recall held, the weakest questions held under rewording, and every head query that could be checked improved. The one honest limit is scope: this is ten tail questions, five head questions, and six paraphrases, all written by the same two people. Real users will ask things none of us thought to test, which is what the pilot is for.

<div style="color:#1e7e34">

## Result: the expansion *improved* retrieval

**Recall did not degrade.** Every question still found its article, Recall@5, @10, and @20 were all unchanged, nothing dropped out of the top 50, and no question fell out of the top 10. The corpus grew by a factor of about 19.5, from 5,687 chunks across 100 articles to 111,048 chunks across 1,803, with no measured loss in retrieval recall.

**The metric that appeared to show degradation turned out not to.** Mean off-article chunks in the top 10 rose from 3.2 to 4.7, and mean rank slipped from 1.7 to 2.2, which initially looked like the crowding mechanism the corpus-scaling research describes. But that metric counts chunks from *other* articles, not from *wrong* ones, and the 100-article slice was alphabetical, so it excluded most of the corpus by construction. Reading the actual results settled it: in all five head queries the additional material was genuinely relevant, and in four of five the full corpus surfaced the entry actually named after the concept where the 100-article corpus had returned articles that merely mentioned it in passing. Supervenience went from nine chunks of *Anomalous Monism* to six of *Supervenience*. Natural kinds went from *Peter Abelard* and *Anaxagoras* to seven chunks of *Natural Kinds*. What looked like crowding was the corpus finally containing the right articles.

**The two ambiguous cases also resolved in favour of the full corpus.** Chunk-level inspection showed the is-ought question retrieving nothing relevant at 100 articles, matching only on the word "ought" in unrelated deontic contexts, while the full corpus led with a passage explicitly on the is-ought gap and scored higher on every result. The a priori question traded some concentration for breadth, and the additions were substantive: a definitional passage from *Epistemology* scoring above anything the 100-corpus produced, plus *Kant's Theory of Judgment* on the synthetic a priori, replacing 100-corpus chunks that had drifted into the Lottery Paradox and other peripheral debates.

**The weakest questions held under rewording.** The al-Farabi set, flagged as least robust, returned ranks 3, 4, 6 on the 100-article corpus and 4, 4, 6 on the full corpus across three phrasings. Rewording moved rank more than expanding the corpus did, which suggests the difficulty is a property of the question rather than of corpus size. The Cicero set held rank 1 across all six runs.

**This result runs against Xiang et al., which is worth stating rather than glossing over.** Their controlled experiments found vector RAG accuracy on complex reasoning dropping 26% relative as the corpus grew 20x, a scale factor almost identical to this expansion's 19.5x, and named the cause as vector retrieval "capturing high-similarity but irrelevant noise as the search space expands." That did not happen here. Several explanations are plausible and not mutually exclusive. Their measurement was end-to-end accuracy on complex reasoning, whereas this one is article-level retrieval recall, which is a looser target that cannot detect within-article failures at all. SEP may also be unusually well-suited to this kind of retrieval, being a curated encyclopedia where most concepts have a dedicated entry, so expansion adds canonical articles rather than more marginally-relevant documents. And the starting corpus here was not a representative sample but an alphabetical slice, so the expansion was partly correcting an artificial deficiency rather than diluting a healthy corpus. The honest position is that this is one result on one corpus with a looser metric, not a refutation.

**Honest limits on what this can claim.** Recall@10 was already at ceiling on the baseline, so that metric could only detect breakage, never improvement, and the improvement had to be found by reading results rather than by any number. The sample is ten tail questions, five head questions, and six paraphrases, all written by the same two people. Real users will ask things none of us thought to test, which is precisely what the pilot is for.

**Decision, per the tree in `scaling_and_pilot_strategy_log.md`:** ship the full corpus, using `query_multi_concat()`, without the lexical phrase-overlap blend. That blend was contingent on degradation that did not appear. It remains available in `10_Oruka_Retrieval_Fix.ipynb` should retrieval quality drop as the corpus grows further.

</div>